# Recommendation_System — Clean Skeleton
This notebook is a condensed, runnable skeleton of the project pipeline. Run cells in order.

In [1]:
import os
import ast
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [2]:
# Load TMDB files (ensure these CSVs are present in the notebook folder)
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")
df = movies.merge(credits.rename(columns={"movie_id": "id"}).drop(columns=["title"]), on="id")
df = df[['id','overview','genres','keywords','title','vote_average','release_date','cast','crew']].copy()

## Exploratory Data Analysis
Run this cell block to generate core EDA plots. (The full notebook contains 10 plots.)

In [3]:
# Example: genre counts
def safe_parse_names(value):
    import ast
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        return [d.get('name','') for d in parsed if isinstance(d, dict)]
    except Exception:
        return []

df['genres_list'] = df['genres'].apply(safe_parse_names)
from collections import Counter
g = Counter([g for sub in df['genres_list'] for g in sub if g])
print('Top genres:', g.most_common(10))

Top genres: [('Drama', 2297), ('Comedy', 1722), ('Thriller', 1274), ('Action', 1154), ('Romance', 894), ('Adventure', 790), ('Crime', 696), ('Science Fiction', 535), ('Horror', 519), ('Family', 513)]


## TF-IDF + Recommendation (memory-safe)
Build TF-IDF over tag fields and compute similarity with linear_kernel.

In [4]:
# Prepare 'tags' column
for col in ['overview','genres','keywords','cast','crew']:
    if col not in df.columns:
        df[col] = ''

df['overview'] = df['overview'].fillna('').astype(str).str.lower()
df['genres'] = df['genres'].fillna('').astype(str).str.lower()
df['keywords'] = df['keywords'].fillna('').astype(str).str.lower()
df['cast'] = df['cast'].fillna('').astype(str).str.lower()
df['crew'] = df['crew'].fillna('').astype(str).str.lower()
df['tags'] = df['overview'] + ' ' + df['genres'] + ' ' + df['keywords'] + ' ' + df['cast'] + ' ' + df['crew']

vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
vectors = vectorizer.fit_transform(df['tags'])
similarity = linear_kernel(vectors, vectors)
print('TF-IDF shape:', vectors.shape, 'Similarity shape:', similarity.shape)
os.makedirs('Model', exist_ok=True)
joblib.dump(vectorizer, 'Model/tfidf_vectorizer.pkl')
joblib.dump(df.to_dict(), 'Model/movies_data.joblib')
joblib.dump(similarity, 'Model/similarity.joblib')

TF-IDF shape: (4803, 5000) Similarity shape: (4803, 4803)


['Model/similarity.joblib']

In [5]:
def recommend(movie_title, top_n=10):
    matches = df[df['title'].str.lower() == movie_title.lower()]
    if matches.empty:
        print(f"Movie '{movie_title}' not found.")
        return []
    idx = matches.index[0]
    distances = similarity[idx]
    scores = sorted(list(enumerate(distances)), key=lambda x: x[1], reverse=True)[1:top_n+1]
    return [df.iloc[i].title for i,_ in scores]

# Example:
print(recommend('The Truman Show', 5))

['Argo', 'Spotlight', 'The Silence of the Lambs', 'The Hobbit: An Unexpected Journey', 'The Jackal']


## Sentiment Analysis (requires IMDB Dataset.csv)
This section is guarded: if `IMDB Dataset.csv` is absent the cells will prompt to add it.

In [ ]:
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

imdb_candidates = ['IMDB Dataset.csv','archive/IMDB Dataset.csv','../archive/IMDB Dataset.csv']
imdb_path = next((p for p in imdb_candidates if os.path.exists(p)), None)

if imdb_path is None:
    print('⚠️  IMDB Dataset.csv not found.')
    print('Download from: https://www.kaggle.com/datasets/marcoiarpa/imdb-review-dataset')
else:
    print('✓ IMDB Dataset found. Training sentiment models...')
    imdb_df = pd.read_csv(imdb_path)
    imdb_df['label'] = (imdb_df['sentiment'] == 'positive').astype(int)
    
    def preprocess(text):
        t = str(text).lower()
        t = re.sub(r'<.*?>','',t)
        t = t.translate(str.maketrans('','',string.punctuation))
        t = re.sub(r'\d+','',t)
        return re.sub(r'\s+',' ',t).strip()
    
    imdb_df['clean_review'] = imdb_df['review'].apply(preprocess)
    tfidf = TfidfVectorizer(max_features=10000, stop_words='english')
    X = tfidf.fit_transform(imdb_df['clean_review'])
    y = imdb_df['label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    lr = LogisticRegression(max_iter=200)
    lr.fit(X_train, y_train)
    preds = lr.predict(X_test)
    
    print('\n=== Logistic Regression Results ===')
    print(f'Accuracy: {accuracy_score(y_test, preds):.4f}')
    print('\nClassification Report:')
    print(classification_report(y_test, preds, target_names=['Negative', 'Positive']))
    
    joblib.dump(lr, 'Model/sentiment_analysis_model.pkl')
    joblib.dump(tfidf, 'Model/tfidf_vectorizer.pkl')
    print('\n✓ Sentiment models saved successfully')